<a href="https://colab.research.google.com/github/melissa-04/melisayla-biyoinformatik/blob/main/notebooks/rna-seq/04_salmon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Salmon ile kantifikasyon: okumalardan sayılara

Bu defterde milyonlarca anonim okumayı genlerine dağıtıp sayacağız. İndeksi beklemeyeceksiniz; kendi kurduğum indeksi Zenodo'dan indiriyoruz. Toplam 10-15 dakika.

In [1]:
import urllib.request
dosyalar = ['SRR384977_1M.fastq.gz', 'SRR384980_1M.fastq.gz', 'salmon_index_vM35.zip.zip']
for d in dosyalar:
    urllib.request.urlretrieve('https://zenodo.org/records/22431622/files/' + d + '?download=1', d)
    print(d, 'indi')

SRR384977_1M.fastq.gz indi
SRR384980_1M.fastq.gz indi
salmon_index_vM35.zip.zip indi


## 1. Salmon'u kurup indeksi açıyorum

Salmon'un hazır derlenmiş 1.10.0 sürümünü kullanıyorum; indeksi de aynı sürümle kurmuştum, uyum şart. (Dosya adındaki çift .zip bir yükleme kazası; içerik sapasağlam, ders: dosya adına değil md5'e güvenin.)

In [2]:
%%bash
set -e
if [ ! -x /content/salmon/bin/salmon ]; then
  wget -q https://github.com/COMBINE-lab/salmon/releases/download/v1.10.0/salmon-1.10.0_linux_x86_64.tar.gz -O salmon.tar.gz
  tar -xzf salmon.tar.gz && mv salmon-latest_linux_x86_64 /content/salmon && rm salmon.tar.gz
fi
unzip -oq salmon_index_vM35.zip.zip
/content/salmon/bin/salmon --version

salmon 1.10.0


In [3]:
import glob, os
IDX = os.path.dirname(glob.glob('**/info.json', recursive=True)[0])
print('İndeks klasörü:', IDX)

İndeks klasörü: salmon_index_vM35


## 2. İki örneği sayıyorum

`-l A`: kütüphane tipini kendin bul demek. Örnek başına birkaç dakika.

In [4]:
for run in ['SRR384977', 'SRR384980']:
    print(run, 'sayılıyor...')
    !/content/salmon/bin/salmon quant -i "{IDX}" -l A -r {run}_1M.fastq.gz -p 2 -o {run}_quant --no-version-check -q
    print(run, 'tamam')

SRR384977 sayılıyor...
-----------------------------------------
| Loading contig table | Time = 326.27 ms
-----------------------------------------
size = 881033
-----------------------------------------
| Loading contig offsets | Time = 9.0504 ms
-----------------------------------------
-----------------------------------------
| Loading reference lengths | Time = 1.985 ms
-----------------------------------------
-----------------------------------------
| Loading mphf table | Time = 263.06 ms
-----------------------------------------
size = 149666271
Number of ones: 881032
Number of ones per inventory item: 512
Inventory entries filled: 1721
-----------------------------------------
| Loading contig boundaries | Time = 393.89 ms
-----------------------------------------
size = 149666271
-----------------------------------------
| Loading sequence | Time = 122.85 ms
-----------------------------------------
size = 123235311
-----------------------------------------
| Loading positi

## 3. Çıktıyı okuyorum: quant.sf

Satırlar transkript. NumReads küsuratlı; Salmon, birden çok transkripte uyan okumaları olasılıkla paylaştırdığı için. Ve eşleşme oranları: FastQC rehberindeki adaptör bulgumuzun faturası burada kesiliyor; hangi örnekte, ne kadar?

In [5]:
import pandas as pd, json

for run in ['SRR384977', 'SRR384980']:
    q = pd.read_csv(f'{run}_quant/quant.sf', sep='\t')
    m = json.load(open(f'{run}_quant/aux_info/meta_info.json'))
    print(f"\n=== {run} — eşleşme: %{m['percent_mapped']:.1f}")
    print(q.sort_values('NumReads', ascending=False).head(5)[['Name','NumReads','TPM']].to_string(index=False))


=== SRR384977 — eşleşme: %66.7
                 Name  NumReads          TPM
 ENSMUST00000093209.4  7830.749 18988.500988
 ENSMUST00000082402.1  7485.469  6293.365851
ENSMUST00000042235.15  5627.737  3968.444083
 ENSMUST00000082405.1  4006.000 10049.742928
ENSMUST00000006749.10  3988.893  1011.167207

=== SRR384980 — eşleşme: %50.2
                 Name  NumReads          TPM
 ENSMUST00000082402.1  7721.760  9417.497588
 ENSMUST00000033229.5  6964.000 28717.656157
ENSMUST00000042235.15  3912.458  4002.132034
 ENSMUST00000082405.1  3856.000 14032.540329
 ENSMUST00000042755.7  3816.183  3324.454796


## Kendin dene

Üç görev: aynı işlemi SRR384982 için yapın. `lib_format_counts.json` dosyasını açıp Salmon'un kütüphane tipi için ne karar verdiğine bakın. Ve en çok okuma alan transkriptlerin adlarını not edin: ENSMUST kodları; hangi genlere ait olduklarını bir sonraki rehberlerde eşleyeceğiz. Eşleşme oranları tam verideki değerlerle (Veri sayfası) birebir aynı çıkmayacak; neden olabilir, düşünün.